In [1]:
import numpy as np
from numpy.linalg import inv as inv
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns
sns.set()

from sklearn.datasets import make_spd_matrix
from sklearn.covariance import graphical_lasso, GraphicalLasso, GraphicalLassoCV
from tqdm.notebook import tqdm

from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
import scipy.cluster.hierarchy as sch
from scipy.stats import chi2
from scipy.stats import multivariate_normal
from scipy.optimize import minimize

from numpy.lib.stride_tricks import sliding_window_view

from statsmodels.stats.multitest import fdrcorrection
from statsmodels.tsa.seasonal import STL

import cvxpy as cp

%matplotlib inline

# fix random seed
np.random.seed(39)

import warnings
warnings.filterwarnings('ignore')  # <- remember to comment this if something breaks and you get confused

In [2]:
def is_pos_def(A):
    if is_symmetric(A):
        try:
            np.linalg.cholesky(A)
            return True
        except np.linalg.LinAlgError:
            return False
    else:
        return False

def is_symmetric(a, tol=1e-8):
    return np.all(np.abs(a-a.T) < tol)

In [3]:
def generate_matrices_orthogonal(prec_coeffs=None, M=2, dim=4):
    H_s = []
    if not prec_coeffs:
        prec_coeffs = np.random.rand(M)
    precision = np.zeros((dim, dim))
    H_s_stacked = np.zeros((dim**2, M))
    mat = make_spd_matrix(dim)
    indxes = np.random.choice(np.arange(dim), (M, dim//M), replace=False)
    for i in range(M):
        A = np.zeros((dim, dim))
        for idx in indxes[i]:
            for idx2 in indxes[i]:
                A[idx, idx2] = mat[idx, idx2]
        H_s.append(A)
        precision += prec_coeffs[i]*A
        H_s_stacked[:, i] = A.flatten()
    H_s = np.array(H_s)
    
    # ensure basis matrices are linearly independent
    assert np.linalg.matrix_rank(H_s_stacked) == M, "Not Linearly Independent basis matrices"
    
    # ensure it's actually symmetric
    # minimal modification on scale of 1e-15
    precision_corrected = (precision + precision.T)/2
    
    return H_s, precision_corrected, prec_coeffs

def collect_precision_matrix(H_s, prec_coeffs):
    precision = (prec_coeffs.reshape(-1, 1, 1)*H_s).sum(0)
    
    return precision

def sim_changepoint_mv_normal_orthogonal(M=2, dim=4, N=500):
    assert dim % M == 0, "Need dim divisible by M for sake of sampling at the moment"
    H_s, precision_one, prec_coeffs_one = generate_matrices_orthogonal(M=M, dim=dim)
    data_one, C_one = sim_data(covar=inv(precision_one), dim=dim, N=N)
    
    prec_coeffs_two = prec_coeffs_one.copy()
    prec_coeffs_two[0] += 0.2
    precision_two = collect_precision_matrix(H_s, prec_coeffs_two)
    data_two, C_two = sim_data(covar=inv(precision_two), dim=dim, N=N)
    
    data_total = np.concatenate((data_one, data_two), axis=1)
    C_total = np.cov(data_total)
    
    return H_s, data_total

def sim_changepoint_mv_normal_orthogonal_mult_coeff(M=2, dim=4, N=500, num_coeffs_change=1):
    assert dim % M == 0, "Need dim divisible by M for sake of sampling at the moment"
    H_s, precision_one, prec_coeffs_one = generate_matrices_orthogonal(M=M, dim=dim)
    data_one, C_one = sim_data(covar=inv(precision_one), dim=dim, N=N)
    
    assert num_coeffs_change <= M, "Cannot change more coefficients than exist"
    
    # multiple coeffs to change - sample them randomly
    to_change_coeffs = np.random.choice(np.arange(M), num_coeffs_change, replace=False)
    prec_coeffs_two = prec_coeffs_one.copy()
    for i in range(num_coeffs_change):
        prec_coeffs_two[to_change_coeffs[i]] += np.random.uniform(0.1, 0.3, 1)[0]
        
    precision_two = collect_precision_matrix(H_s, prec_coeffs_two)
    data_two, C_two = sim_data(covar=inv(precision_two), dim=dim, N=N)
    
    data_total = np.concatenate((data_one, data_two), axis=1)
    C_total = np.cov(data_total)
    print("Precision Coefficients Pre-Changepoint: ", prec_coeffs_one)
    print("Precision Coefficients Post-Changepoint: ", prec_coeffs_two)
    
    return H_s, data_total

def sim_data(covar, dim, N=1000):
    assert is_symmetric(covar), is_pos_def(covar)
    data_sim = np.random.multivariate_normal(np.zeros(dim), covar, N).T
    data_sim = data_sim - data_sim.mean()
    C = np.cov(data_sim)
    
    return data_sim, C

In [4]:
M = 2
dim = 4
N = 1000

H_s, precision, prec_coeffs = generate_matrices_orthogonal(M=M, dim=dim)
data_full, C_full = sim_data(covar=inv(precision), dim=dim, N=N)
C_full.shape

(4, 4)

In [13]:
alphas = cp.Variable(shape=prec_coeffs.shape)
lam = 1e-2
psi_hat = sum([alphas[i]*H_s[i] for i in range(M)])
l1_penalty = sum([cp.abs(psi_hat[i, j])
                  for i in range(dim)
                  for j in range(dim) if i != j])
objective = cp.Maximize(cp.log_det(psi_hat) - cp.trace(psi_hat@C_full) - lam*l1_penalty)
constr1 = (psi_hat >> 0)
constr2 = (psi_hat == psi_hat.T)
problem = cp.Problem(objective)
problem.solve()
if problem.status != cp.OPTIMAL:
    raise Exception('CVXPY Error')
alphas.value

array([0.51770568, 0.80407714])

In [14]:
prec_coeffs

array([0.54688916, 0.79789902])

In [22]:
url = 'https://raw.githubusercontent.com/empathy87/The-Elements-of-Statistical-Learning-Python-Notebooks/master/data/protein.data'
df = pd.read_csv(url, header=None, sep=' ')

X = df.to_numpy()

protein_names = ['Raf', 'Mek', 'Plcg', 'PIP2', 'PIP3', 'Erk', 'Akt', 'PKA', 'PKC', 'P38', 'Jnk']
p = len(protein_names)

# the empirical covariance matrix
S = np.cov(X, rowvar=False)/1000
lambdas = [36, 27, 7]
theta_estimates = []
S.shape

(11, 11)

In [25]:
# theta should be symmetric positive-definite
theta = cp.Variable(shape=(p, p), PSD=True)
# An alternative formulation of the problem () can be posed,
# where we don't penalize the diagonal of theta.
lam = 2.0
l1_penalty = sum([cp.abs(theta[i, j])
                  for i in range(p)
                  for j in range(p) if i != j])
objective = cp.Maximize(cp.log_det(theta) - cp.trace(theta@S) - lam*l1_penalty)
problem = cp.Problem(objective)
problem.solve()
if problem.status != cp.OPTIMAL:
    raise Exception('CVXPY Error')
theta.value

array([[ 2.65781064e-01, -1.69036737e-01, -4.72320649e-08,
        -5.09790723e-09, -4.92196663e-09, -1.22812676e-08,
        -1.37120952e-07,  2.90352506e-09, -1.09319672e-08,
        -1.42327818e-12,  5.89667492e-09],
       [-1.69036737e-01,  1.15493695e-01, -2.04597524e-03,
        -2.30031145e-08,  2.71666676e-09, -1.50586209e-08,
        -4.06618401e-03,  4.96759475e-04,  8.36957776e-09,
        -6.33939500e-04,  2.27002038e-08],
       [-4.72320649e-08, -2.04597524e-03,  1.59884130e-01,
        -7.92052958e-02,  6.83821892e-09, -7.51847651e-09,
        -5.53234880e-03,  8.79596857e-04, -2.05713106e-09,
        -1.71530320e-03, -1.76531671e-03],
       [-5.09790723e-09, -2.30031145e-08, -7.92052958e-02,
         5.33896716e-02, -3.09762263e-03,  1.43038036e-09,
        -4.62391430e-03,  2.39247393e-04,  2.78024132e-10,
         1.08034938e-09, -2.04791605e-03],
       [-4.92196663e-09,  2.71666676e-09,  6.83821892e-09,
        -3.09762263e-03,  5.40482356e-01,  8.63904275e-11,
  